# DCPA on NVIDIA DGX Spark（GB10 Grace Blackwell · 128 GB 统一内存）一键全科发育工作台

本 Notebook 专为 **NVIDIA DGX Spark**（ARM64 `aarch64` + GB10 Blackwell GPU + 128 GB LPDDR5X 统一内存 + 本地高速 NVMe）定制，默认启用 **DCPA v2 (`moe_210m`) 五阶段全科认知发育课程（35% 逻辑与通识推演 : 35% 深度写作 : 30% 工程代码，零 Rust）**，从 Cell 1 顺序执行到 Cell 8 即可完成 `~5.9 GB` 七路并行构建与校验 → `100,000` 步底座训练（`[0, 12k, 42k, 78k, 100k]`） → `15,000` 步苏格拉底思辨与多稿精修 SFT 微调 → 导出至 Mac M3 Pro。

---

## 第 0 步：首次在 DGX Spark 上启动本 Notebook（仅需做一次，二选一）

先把本代码仓库传到 DGX Spark（`git clone` 或 `scp`），在 DGX Spark 终端进入仓库根目录后执行：

### 方案 A（强烈推荐：NVIDIA 官方 NGC PyTorch 容器，原生内置 GB10 Blackwell CUDA/Triton/bf16）
```bash
bash scripts/setup_dgx_spark.sh --ngc
```

### 方案 B（直接在 DGX OS 宿主机建独立 Python venv，自动装 ARM64 `cu128` PyTorch）
```bash
bash scripts/setup_dgx_spark.sh --native
```

### 在 Mac 浏览器打开 JupyterLab（SSH 端口转发）
```bash
ssh -L 8888:localhost:8888 <你的用户名>@<DGX_Spark_IP>
# 然后在 Mac 浏览器打开 http://localhost:8888 -> 点击 notebooks/DGX_Spark_DCPA_Train.ipynb
```

---

## DCPA v2 (`v2_cognitive_5g` 档位) 七路并行数据清单（总计约 `5.9 GB`，零 Rust）

| 序号 | 落盘分片 (`artifacts/`) | 目标体积 | 数据源与炼金术作用 |
|---|---|---|---|
| 1 | `data/{train,val}.bin/.idx` | `~5.28 GB` | 六域分级底座语料（`0=en` FinePhrase/US-PD-Books/Cosmopedia / `1=zh` FineWeb-Edu-ZH/CCI3-HQ / `2=formal` FOLIO+ProofWriter+LogiQA+FineMath / `3=python` python-edu+swallow-code-v2 / `4=sql_struct` SQL+JSON / `5=javascript` ES6+ 模块化），写入 `0..4` 正交 `style_id` |
| 2 | `cap_data/cap_reasoning_*` | `160 MB` | `PleIAs/SYNTH`（`→/↺/∴/●` 速记）+ `natural_reasoning` + `COIG-CQIA` + 13 类逻辑谬误诊断 + `40%` 潜空间蒸馏 |
| 3 | `cap_data/cap_writing_*` | `160 MB` | `Crownelius` 叙事子集 + 逆向提纲锁 `Slot-0` + 25% 去 AI 腔病文改好文重构对（`slop [mask=0] -> 诊断+定稿 [mask=1]`） |
| 4 | `cap_data/cap_code_*` | `120 MB` | `OpenCodeReasoning-2` + `commitpackft` + 先骨架后编码（`guard\|bp\|flow`）+ 坏味道重构对比（Python / SQL / ES6 JS，`tree-sitter` 零错误） |
| 5 | `cap_data/cap_instruct_*` | `40 MB` | 四域严格对齐指令遵循（长程约束、JSON Schema、结构化摘要） |
| 6 | `cap_data/cap_toolcall_*` | `40 MB` | 海马体单行工具签名 + `30%` OOV 生造别名 + 链式调用 / 报错自愈 / 零调用克制 |
| 7 | `sft_data/sft_{train,val}.*` | `84 MB` | Stage 5 认知 SFT（`35%` 苏格拉底反诘与边界拒答 : `35%` 作者多稿打磨闭环 : `30%` 仓内 Bug 排错与合约演化，`<=4096 B`） |


In [ ]:
# Cell 1 [全局配置开关 —— 只需按需修改这里，下面所有单元格直接无脑顺序运行]
import os
import sys
from pathlib import Path

# ==============================================================================
# 1. 模型规模开关（DGX Spark 拥有 128 GB 统一内存，推荐默认 "210m" 全科发育版！）
#    "210m" -> DCPA v2 全科认知 35:35:30（208.65M 总参 / 22.86M 激活，32专家 Top-2，零 Rust）
#    "48m"  -> DCPA v1 基线（47.62M 总参 / 15.77M 激活，8专家 Top-2）
# ==============================================================================
MODEL_PRESET = "210m"

# ==============================================================================
# 2. 快速全链路冒烟模式开关：
#    True  -> 使用 v2_cognitive_smoke (~22 MB)，底座跑 200 步 + Stage 5 跑 50 步（约 2 分钟验证整机无坑）
#    False -> 正式全量：v2_cognitive_5g (~5.9 GB) + 底座 100,000 步 + Stage 5 15,000 步
# ==============================================================================
SMOKE_TEST_ONLY = False

# ==============================================================================
# 3. 可选高级项（一般保持默认即可）
# ==============================================================================
COMPILE_MODE = "auto"         # "auto" | "reduce-overhead" | "none"（若极早期驱动 Triton 报错可改为 "none"）
AUTO_GIT_PULL = False         # True = 每次执行自动 `git pull --ff-only` 同步最新代码
USE_WANDB = True              # True = 开启 Weights & Biases 实验追踪（未设 WANDB_API_KEY 自动走 offline）
WANDB_PROJECT = "dcpa-turbo"
FORCE_REBUILD_DATA = False    # True = 强制删除旧数据重新流式拉取；False = 已有分片自动秒跳过
MAX_PARALLEL_DATA_JOBS = 7    # 七路并行数据构建并发数
HF_TOKEN = os.environ.get("HF_TOKEN", "")        # 可选：填 HuggingFace Token 可解锁 gated CCI3-HQ / xLAM
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "")  # 可选：国内加速镜像例如 "https://hf-mirror.com"

# 自动定位仓库根目录（无论在根目录还是 notebooks/ 子目录启动 JupyterLab 都能精准找到）
cwd = Path.cwd().resolve()
if (cwd / "train" / "train_dcpa.py").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "train" / "train_dcpa.py").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"未找到 DCPA 仓库根目录，当前目录: {cwd}")

os.chdir(REPO_ROOT)
if AUTO_GIT_PULL:
    import subprocess as _sp
    _sp.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
for _p in (str(REPO_ROOT), str(REPO_ROOT / "dcpa-train-py")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

os.environ["PYTHONPATH"] = f"{REPO_ROOT}:{REPO_ROOT / 'dcpa-train-py'}"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["DCPA_MODEL_PRESET"] = MODEL_PRESET
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
if HF_ENDPOINT:
    os.environ["HF_ENDPOINT"] = HF_ENDPOINT

if MODEL_PRESET == "210m":
    CONFIG_YAML = str(REPO_ROOT / "configs" / "v2" / "dcpa_210m_v2_cognitive.yaml")
    PRESET_NAME = "moe_210m"
    DATA_PROFILE = "v2_cognitive_smoke" if SMOKE_TEST_ONLY else "v2_cognitive_5g"
elif MODEL_PRESET == "48m":
    CONFIG_YAML = str(REPO_ROOT / "configs" / "v1" / "dcpa_48m_train.yaml")
    PRESET_NAME = "proto_48m"
    DATA_PROFILE = "smoke" if SMOKE_TEST_ONLY else "v1_fast"
else:
    raise ValueError(f"MODEL_PRESET 必须为 '48m' 或 '210m'，当前: {MODEL_PRESET!r}")

NVME_ROOT = REPO_ROOT / "artifacts"
DATA_DIR = NVME_ROOT / "data"
CAP_DATA_DIR = NVME_ROOT / "cap_data"
SFT_DATA_DIR = NVME_ROOT / "sft_data"
CKPT_DIR = NVME_ROOT / "checkpoints"
LOGS_DIR = NVME_ROOT / "logs"
EXPORT_DIR = NVME_ROOT / "export"
for _d in (DATA_DIR, CAP_DATA_DIR, SFT_DATA_DIR, CKPT_DIR, LOGS_DIR, EXPORT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"[Config] REPO_ROOT       = {REPO_ROOT}")
print(f"[Config] MODEL_PRESET    = {MODEL_PRESET} ({PRESET_NAME} -> {Path(CONFIG_YAML).name})")
print(f"[Config] DATA_PROFILE    = {DATA_PROFILE} | SMOKE_TEST_ONLY = {SMOKE_TEST_ONLY} | COMPILE_MODE = {COMPILE_MODE}")
print(f"[Config] HF_ENDPOINT     = {HF_ENDPOINT or '(official huggingface.co)'} | HF_TOKEN={'SET' if HF_TOKEN else 'not set (ungated sources OK)'}")


In [ ]:
# Cell 2 [DGX Spark 硬件/CUDA 体检 + 全套依赖一键安装]
import os
import platform
import shutil
import subprocess
import sys

print(f"[Host] Arch={platform.machine()} | OS={platform.platform()} | Python={sys.version.split()[0]}")
if hasattr(os, "sysconf"):
    try:
        ram_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES")) / (1024 ** 3)
        print(f"[Host] Unified/System RAM = {ram_gb:.1f} GB")
    except Exception:
        pass
disk = shutil.disk_usage(REPO_ROOT)
print(f"[Disk] Free NVMe space at {REPO_ROOT}: {disk.free / (1024 ** 3):.1f} GB")

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
if smi.returncode == 0:
    print("\n".join(smi.stdout.splitlines()[:14]))
else:
    print("[WARN] nvidia-smi 未返回 GPU 信息，请确认已用 --gpus all 启动容器或在 DGX OS 主机运行！")

# 1. 检查 PyTorch + CUDA（若未装 CUDA 版则自动拉取 ARM64 cu128 轮子）
need_torch_install = False
try:
    import torch
    if not torch.cuda.is_available():
        print("[WARN] 当前 PyTorch 未启用 CUDA，尝试安装 CUDA 12.8 (Blackwell) 适配版 PyTorch...")
        need_torch_install = True
except ImportError:
    need_torch_install = True

if need_torch_install:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade", "torch", "torchvision",
         "--index-url", "https://download.pytorch.org/whl/cu128"],
        check=True,
    )
    import importlib
    import torch
    importlib.reload(torch)

# 2. 一键安装全部数据流式下载、Python/JS tree-sitter AST 过滤与可视化依赖
deps = [
    "safetensors",
    "datasets",
    "hf_transfer",
    "wandb",
    "tree-sitter",
    "tree-sitter-python",
    "tree-sitter-javascript",
    "tree-sitter-rust",
    "pyyaml",
    "pytest",
    "matplotlib",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(REPO_ROOT / "dcpa-train-py")],
    check=True,
)

# 3. 打印 DCPA 跨平台设备层最终识别结果与模型参数量
from dcpa.model import DCPAModel
from train.train_dcpa import build_dcpa_config, load_yaml_config
from utils.device import describe_hardware_profile, detect_hardware_profile

profile = detect_hardware_profile("auto")
print("\n" + describe_hardware_profile(profile))
if profile.device_type != "cuda":
    raise RuntimeError(
        "PyTorch 仍未识别到 DGX Spark GPU！请先用官方 NGC 容器启动：bash scripts/setup_dgx_spark.sh --ngc"
    )

cfg_preview = build_dcpa_config(PRESET_NAME, load_yaml_config(CONFIG_YAML))
m_counts = DCPAModel(cfg_preview).count_parameters()
print(
    f"[Model Check] {PRESET_NAME} ({Path(CONFIG_YAML).name}): total={m_counts['total']:,} ({m_counts['total']/1e6:.2f}M) | "
    f"active/step={m_counts['active_per_step']:,} ({m_counts['active_per_step']/1e6:.2f}M) | "
    f"E={cfg_preview.num_experts} top_k={cfg_preview.top_k_experts} L_max={cfg_preview.l_max} W_max={cfg_preview.householder_max_writes}"
)


In [ ]:
# Cell 3 [七路并行纯字节语料一键流式构建与校验（v2_cognitive_5g 约 5.9 GB / 已有分片自动跳过）]
import subprocess
import sys
import time

t_all = time.perf_counter()
prep_cmd = [
    sys.executable,
    "-u",
    "data/prepare_vast_corpus.py",
    "--profile", DATA_PROFILE,
    "--nvme-root", str(NVME_ROOT),
    "--max-parallel-jobs", str(MAX_PARALLEL_DATA_JOBS),
]
if FORCE_REBUILD_DATA:
    prep_cmd.append("--force")

print(f"[Data Orchestrator] profile={DATA_PROFILE} | Running: {' '.join(prep_cmd)}", flush=True)
subprocess.run(prep_cmd, check=True)

# 汇总所有数据分片体积
print("\n" + "=" * 72)
print(f"[Data Ready] 全部语料校验通过（总耗时 {time.perf_counter() - t_all:.1f}s）：")
total_mb = 0.0
for p in sorted([*DATA_DIR.glob("*.bin"), *CAP_DATA_DIR.glob("*.bin"), *SFT_DATA_DIR.glob("*.bin")]):
    if p.name.endswith(".loss_mask.bin"):
        continue
    mb = p.stat().st_size / (1024 ** 2)
    total_mb += mb
    print(f"  - {str(p.relative_to(REPO_ROOT)):44s} : {mb:7.2f} MB")
print(f"  >>> 纯字节文本总量合计: {total_mb:.2f} MB")
print("=" * 72)


In [ ]:
# Cell 4 [DGX Spark GB10 上线全套复检：因果律 / STE / 数据纯净度 / DCPA v2 认知管线 / GPU 吞吐]
import os
import subprocess
import sys
import time

t0 = time.perf_counter()
env = dict(os.environ)
env["DCPA_DATA_DIR"] = str(DATA_DIR)

test_files = [
    "tests/test_sanity.py",
    "tests/test_data_pipeline.py",
    "tests/test_capabilities_upgrade.py",
    "tests/test_v2_cognitive_pipeline.py",
    "tests/test_perf_and_compile.py",
]
subprocess.run([sys.executable, "-m", "pytest", "-v", "-s", *test_files], env=env, check=True)
print(f"\n[DGX Spark Preflight] 全套 5 组硬核验收 + GPU 吞吐基准 100% GREEN（耗时 {time.perf_counter() - t0:.1f}s）!")


In [ ]:
# Cell 5 [一键启动 / 断电自动续跑 100,000 步底座训练（Stage 1 -> Stage 4 全科认知发育）]
# ==============================================================================
# 提示：若希望关掉电脑通宵后台跑，可直接复制下方打印出的 [Train Command] 到 DGX Spark 的 tmux 中运行：
#   tmux new -s dcpa_v2
#   <粘贴命令回车>
#   按 Ctrl+B 然后按 D 后台挂起；随时 tmux attach -t dcpa_v2 查看进度！
# ==============================================================================
import os
import subprocess
import sys

total_steps = 200 if SMOKE_TEST_ONLY else 100000
save_every = 50 if SMOKE_TEST_ONLY else 1000
probe_every = 50 if SMOKE_TEST_ONLY else 500
stage_bounds_args = ["--stage-boundaries", "0,30,90,150,200"] if SMOKE_TEST_ONLY else []
seq_args = [] if MODEL_PRESET == "210m" else ["--seq-len", "2048"]

train_cmd = [
    sys.executable,
    "-u",
    "train/train_dcpa.py",
    "--gpus", "1",
    "--config", CONFIG_YAML,
    "--preset", PRESET_NAME,
    "--total-steps", str(total_steps),
    *stage_bounds_args,
    *seq_args,
    "--data-dir", str(DATA_DIR),
    "--cap-data-dir", str(CAP_DATA_DIR),
    "--ckpt-dir", str(CKPT_DIR),
    "--logs-dir", str(LOGS_DIR),
    "--export-dir", str(EXPORT_DIR),
    "--local-cache-dir", str(NVME_ROOT),
    "--save-every-steps", str(save_every),
    "--save-every-minutes", "10",
    "--probe-every-steps", str(probe_every),
    "--heartbeat-seconds", "60",
    "--resume", "auto",
    "--compile-mode", COMPILE_MODE,
    "--max-consecutive-meltdowns", "3",
]
if USE_WANDB:
    train_cmd.extend(["--wandb", "--wandb-project", WANDB_PROJECT])

print("[Train Command] " + " ".join(train_cmd), flush=True)
log_file = LOGS_DIR / "spark_stage1_4_stdout.log"
with log_file.open("a", encoding="utf-8") as lf:
    proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        lf.write(line)
    exit_code = int(proc.wait())

if hasattr(os, "sync"):
    os.sync()
base_sf = EXPORT_DIR / "dcpa_final.safetensors"
if exit_code == 0 and base_sf.exists():
    print(f"\n[Stage 1-4 DONE] 底座训练完工！导出权重: {base_sf} ({base_sf.stat().st_size / (1024**2):.2f} MB)")
elif exit_code == 42:
    raise RuntimeError("[CIRCUIT_BREAKER] 连续 3 次看门狗回滚仍未恢复，请检查 artifacts/logs/watchdog_events.jsonl")
else:
    raise RuntimeError(f"底座训练异常退出，exit_code={exit_code}")


In [ ]:
# Cell 6 [一键启动 Stage 5：15,000 步全科多稿打磨 / 苏格拉底思辨 / 仓内排错与工具微调]
import os
import subprocess
import sys

BASE_WEIGHTS = EXPORT_DIR / "dcpa_final.safetensors"
BASE_CONFIG = EXPORT_DIR / "config.json"
CHAT_WEIGHTS = EXPORT_DIR / "dcpa_chat_sft.safetensors"
if not BASE_WEIGHTS.exists():
    raise FileNotFoundError(f"未找到底座权重 {BASE_WEIGHTS}，请先运行 Cell 5 完成 Stage 1~4 训练！")

sft_steps = 50 if SMOKE_TEST_ONLY else (15000 if MODEL_PRESET == "210m" else 8000)
max_doc_b = 4096 if MODEL_PRESET == "210m" else 2048
sft_cmd = [
    sys.executable,
    "-u",
    "train/train_stage5_sft.py",
    "--gpus", "1",
    "--weights", str(BASE_WEIGHTS),
    "--data-dir", str(SFT_DATA_DIR),
    "--cap-data-dir", str(CAP_DATA_DIR),
    "--export-dir", str(EXPORT_DIR),
    "--total-steps", str(sft_steps),
    "--target-tokens-bytes-per-step", "65536",
    "--max-doc-bytes", str(max_doc_b),
    "--compile-mode", COMPILE_MODE,
    "--log-every", "25" if SMOKE_TEST_ONLY else "50",
]
if BASE_CONFIG.exists():
    sft_cmd.extend(["--config", str(BASE_CONFIG)])
if USE_WANDB:
    sft_cmd.extend(["--wandb", "--wandb-project", WANDB_PROJECT])

print("[Stage 5 Command] " + " ".join(sft_cmd), flush=True)
subprocess.run(sft_cmd, check=True)
if hasattr(os, "sync"):
    os.sync()
print(f"\n[Stage 5 DONE] 最终对话与全科微调权重已导出: {CHAT_WEIGHTS} ({CHAT_WEIGHTS.stat().st_size / (1024**2):.2f} MB)")


In [ ]:
# Cell 7 [训练曲线可视化 + 最新视觉探针报告 + 一键传回 Mac M3 Pro 命令]
import json
import math
import socket

metrics_path = LOGS_DIR / "metrics.jsonl"
if metrics_path.exists():
    rows = [json.loads(line) for line in metrics_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"[Telemetry] 共读取 {len(rows)} 条训练记录 ({metrics_path})")
    if rows:
        steps = [r.get("step", i + 1) for i, r in enumerate(rows)]
        losses = [r.get("loss_total", 0.0) for r in rows]
        bpcs = [r.get("loss_lang", 0.0) / math.log(2.0) for r in rows]
        rhos = [r.get("compression_ratio", 0.0) for r in rows]
        try:
            import matplotlib.pyplot as plt
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            axes[0].plot(steps, losses); axes[0].set_title("Total Loss"); axes[0].set_xlabel("Step")
            axes[1].plot(steps, bpcs, color="tab:orange"); axes[1].set_title("Language BPC"); axes[1].set_xlabel("Step")
            axes[2].plot(steps, rhos, color="tab:green"); axes[2].set_title("SGSC Compression rho"); axes[2].set_xlabel("Step")
            plt.tight_layout()
            plt.show()
        except Exception as exc:
            print(f"[Plot Skipped] {exc} | Latest step={steps[-1]}, loss={losses[-1]:.4f}, bpc={bpcs[-1]:.3f}, rho={rhos[-1]:.2f}x")
else:
    print("[Telemetry] 尚未生成 metrics.jsonl（运行 Cell 5 后自动产出）。")

probe_md = LOGS_DIR / "latest_probe.md"
if probe_md.exists():
    print("\n" + "-" * 72 + "\n[Latest Visual Probe]\n" + "-" * 72)
    print(probe_md.read_text(encoding="utf-8")[:3000])

chat_sf = EXPORT_DIR / "dcpa_chat_sft.safetensors"
cfg_js = EXPORT_DIR / "config.json"
host = socket.gethostname()
print("\n" + "=" * 72)
print("[回传 Mac M3 Pro 本地极速交互终端命令]")
print(f"1. 在 Mac 终端拷贝权重 (约 {chat_sf.stat().st_size / (1024**2):.1f} MB):" if chat_sf.exists() else "1. 训完后在 Mac 终端拷贝权重:")
print(f"   scp <user>@{host}:{EXPORT_DIR}/dcpa_chat_sft.safetensors <user>@{host}:{EXPORT_DIR}/config.json ./artifacts/export/")
print("2. 在 Mac M3 Pro 启动本地 O(1) 对话 + 安全 AST 工具终端:")
print("   python cli/chat_m3.py --weights artifacts/export/dcpa_chat_sft.safetensors --device mps")
print("=" * 72)


In [ ]:
# Cell 8 [在 DGX Spark 上直接加载刚训完的权重试聊 + 实测 6x5 正交文风切换与海马体工具沙箱]
from cli.chat_m3 import DCPAChatSession, execute_tool_call_safe
from train.train_stage5_sft import load_dcpa_model_from_weights
from utils.device import get_device

weights_to_test = EXPORT_DIR / "dcpa_chat_sft.safetensors"
if not weights_to_test.exists():
    weights_to_test = EXPORT_DIR / "dcpa_final.safetensors"
if not weights_to_test.exists():
    raise FileNotFoundError("尚未找到导出权重，请先跑完 Cell 5 / Cell 6！")

dev = get_device("auto")
chat_model = load_dcpa_model_from_weights(weights_to_test, device=dev)
session = DCPAChatSession(chat_model, device=dev, temp=0.3, top_p=0.9, max_new_bytes=192)
print(f"[Loaded] {weights_to_test.name} on {dev} | 预置海马体工具数: {len(session.tool_signatures)}")

# 验证安全 AST 工具沙箱（calc / now_hk / 恶意拦截）
print("[Sandbox calc]  ", execute_tool_call_safe('calc("(128 * 3 + 16) / 4")', session.tool_registry))
print("[Sandbox now_hk]", execute_tool_call_safe('now_hk()', session.tool_registry))
print("[Sandbox block] ", execute_tool_call_safe("__import__('os').system('id')", session.tool_registry))

# 验证 PMD 6x5 正交风格切换与 O(1) 海马体快权重跨轮记忆
for style_cmd, prompt in (
    ("/style memo", "记住新概念：‘星曜缓存’是一种固定8槽的零KV内存索引。请用一句话总结它的优势。"),
    ("/style memo", "刚才提到的星曜缓存有多少个固定槽位？"),
    ("/style narrative", "写两句傍晚老船坞边修灯塔的短景，克制写实。"),
):
    session.handle_command(style_cmd)
    reply, tel = session.chat_turn(prompt)
    print(f"\n[{style_cmd}] you > {prompt}")
    print(f"dcpa> {reply}")
    print(f"      {tel.format_line()}")
